# EmPath: 2021 canopy for the pilot ZIP areas
Run this notebook in VS Code using the **Python (EmPath)** kernel. It uses the existing project script and boundary polygons.

Scope: **75025, 75024, 75074, and comparison 75023**, clipped to Collin County. Set `FULL_ZCTAS=True` to keep their full footprints.

**Local download:** the first run downloads the ~2.25 GB Texas archive to your laptop and extracts about 20 GB. Allow at least 30 GB free space. It can take as long as your original download; clipping reduces the output size, not the initial download. Later runs reuse the cached archive. An interrupted download restarts on the next run. To use an already extracted raster, set `SOURCE_RASTER` below and keep its companion `.ige` in the same folder.

[Source: Texas 2021 canopy, 10 m](https://zenodo.org/records/8200761). Canopy cover is not time-specific sidewalk shade. ZCTAs approximate postal ZIPs.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts/clip_texas_canopy.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/clip_texas_canopy.py').exists(), 'Open this notebook inside the EmPath project.'
print('Project:', ROOT)
print('Python:', sys.executable)

ZIPCODES = ['75025', '75024', '75074', '75023']
FULL_ZCTAS = False
SOURCE_RASTER = None  # Or Path('/path/to/TX_10m_canopy_cover_2021.img')
OUTPUT = ROOT / 'output/canopy_2021'


Project: /Users/nalinikantheti/Documents/ChatGPT/EmPath
Python: /Users/nalinikantheti/Documents/ChatGPT/EmPath/.venv/bin/python


In [3]:
# The subprocess uses this notebook's Python environment.
# Remove inherited Anaconda data paths so Rasterio uses its bundled databases.
env = os.environ.copy()
for key in ('PROJ_DATA', 'PROJ_LIB', 'GDAL_DATA'):
    env.pop(key, None)
subprocess.run([sys.executable, '-c', 'import rasterio, geopandas, matplotlib; print("Canopy dependencies ready")'],
               env=env, check=True)
free_gb = shutil.disk_usage(ROOT).free / 1e9
print(f'Free disk: {free_gb:.1f} GB')
cache = ROOT / 'data/raw/texas_canopy_2021'
if SOURCE_RASTER is None and not (cache / 'TX_10m_canopy_cover_2021.img').exists() and free_gb < 30:
    raise RuntimeError('Allow at least 30 GB free before downloading and extracting the statewide data.')


Canopy dependencies ready
Free disk: 119.6 GB


In [4]:
# First run downloads the statewide archive. Progress appears below.
command = [sys.executable, '-u', str(ROOT / 'scripts/clip_texas_canopy.py'),
           '--boundaries', str(ROOT / 'output/scope_screen/zcta_boundaries.geojson'),
           '--cache', str(cache), '--output', str(OUTPUT), '--zipcodes', *ZIPCODES]
if SOURCE_RASTER is None:
    command.append('--download')
else:
    command.extend(['--source-raster', str(SOURCE_RASTER)])
if FULL_ZCTAS:
    command.append('--full-zctas')
subprocess.run(command, cwd=ROOT, env=env, check=True)


Downloaded 8 MB
Downloaded 17 MB
Downloaded 25 MB
Downloaded 34 MB
Downloaded 42 MB
Downloaded 50 MB
Downloaded 59 MB
Downloaded 67 MB
Downloaded 75 MB
Downloaded 84 MB
Downloaded 92 MB
Downloaded 101 MB
Downloaded 109 MB
Downloaded 117 MB
Downloaded 126 MB
Downloaded 134 MB
Downloaded 143 MB
Downloaded 151 MB
Downloaded 159 MB
Downloaded 168 MB
Downloaded 176 MB
Downloaded 185 MB
Downloaded 193 MB
Downloaded 201 MB
Downloaded 210 MB
Downloaded 218 MB
Downloaded 226 MB
Downloaded 235 MB
Downloaded 243 MB
Downloaded 252 MB
Downloaded 260 MB
Downloaded 268 MB
Downloaded 277 MB
Downloaded 285 MB
Downloaded 294 MB
Downloaded 302 MB
Downloaded 310 MB
Downloaded 319 MB
Downloaded 327 MB
Downloaded 336 MB
Downloaded 344 MB
Downloaded 352 MB
Downloaded 361 MB
Downloaded 369 MB
Downloaded 377 MB
Downloaded 386 MB
Downloaded 394 MB
Downloaded 403 MB
Downloaded 411 MB
Downloaded 419 MB
Downloaded 428 MB
Downloaded 436 MB
Downloaded 445 MB
Downloaded 453 MB
Downloaded 461 MB
Downloaded 470 MB
Down

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/urllib3/response.py", line 748, in _error_catcher
    yield
  File "/opt/anaconda3/lib/python3.12/site-packages/urllib3/response.py", line 873, in _raw_read
    data = self._fp_read(amt, read1=read1) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/urllib3/response.py", line 856, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/http/client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/socket.py", line 708, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/ssl.py", line 1252, in recv_into
    return self.read(nbytes, buffer)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.1

CalledProcessError: Command '['/Users/nalinikantheti/Documents/ChatGPT/EmPath/.venv/bin/python', '-u', '/Users/nalinikantheti/Documents/ChatGPT/EmPath/scripts/clip_texas_canopy.py', '--boundaries', '/Users/nalinikantheti/Documents/ChatGPT/EmPath/output/scope_screen/zcta_boundaries.geojson', '--cache', '/Users/nalinikantheti/Documents/ChatGPT/EmPath/data/raw/texas_canopy_2021', '--output', '/Users/nalinikantheti/Documents/ChatGPT/EmPath/output/canopy_2021', '--zipcodes', '75025', '75024', '75074', '75023', '--download']' returned non-zero exit status 1.

In [2]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'canopy_preview.png')))
print((OUTPUT / 'source_metadata.json').read_text())
print('Saved files:')
for path in sorted(OUTPUT.iterdir()):
    print(f'{path.name}: {path.stat().st_size / 1e6:.2f} MB')


NameError: name 'OUTPUT' is not defined

The preview and GeoTIFFs are saved in `output/canopy_2021/`. The combined raster is `canopy_scope_2021.tif`; individual files are named by ZCTA. The notebook preserves native raster resolution and masks outside-scope/source NoData pixels. The existing pilot scope and data cleaning notebooks remain separate.